# 05 · Phenoconversion-free survival by SAA status and SAA kinetics

One figure (Kaplan–Meier curves with numbers at risk, and a forest plot of adjusted hazard ratios):

- **A** – all participants in the survival set: SAA-positive vs SAA-negative (reference)
- **B** – adjusted HRs (Cox, adjusted for age and sex): SAA status, and Fmax, TTT and AUC
- **D–F** – SAA-positive participants with Project 237 (24-hour assay) kinetics: median split of Fmax (High vs Low), TTT (Fast vs Slow) and AUC (High vs Low)

Event: first visit with PRIMDIAG = 1 (PD) or 5 (DLB); otherwise censored at the last visit. Prevalent cases and baseline-only participants are excluded; data are censored at 12 years.

**Inputs:** `results_baseline/prodromal_saa_longitudinal.csv` and `prodromal_saa_baseline_core.csv` (notebook 01)

**Outputs** (`analysis/figures/`): `fig4b_KM_kinetics_median_split_with_forest.png` / `.pdf` / `.tiff`, `fig4b_kinetics_median_split_estimates.csv`

In [ ]:
# Works in Google Colab (mounts Drive) and locally. Set the project folder here,
# or through the PPMI_SAA_BASE environment variable.
# Needs lifelines (not preinstalled in Colab): %pip install lifelines
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE_DIR = os.environ.get('PPMI_SAA_BASE',
                          '/content/drive/MyDrive/PPMI_Projects/prodromal_SAA_groups')

## Configuration

In [ ]:
# ------------------------------- CONFIG -------------------------------------
# --- Sample definition (survival set) --------------------------------------
CORE_ENROLL_YEAR_MAX   = None    # ascertainment filter; None to disable
CORE_EXCLUDE_PREVALENT = True    # drop participants already diagnosed at baseline
CORE_REQUIRE_FOLLOWUP  = True    # survival needs >=1 post-baseline visit

# --- Outcome ----------------------------------------------------------------
# PPMI PRIMDIAG: 1 = idiopathic PD, 5 = DLB, 11 = MSA. Codes 23/24/25 are
# prodromal designations, NOT conversion -- they must never be events.
CORE_EVENT_PRIMDIAG = [1, 5]
CORE_DIAG_NAMES     = {1: 'PD', 5: 'DLB'}
CORE_TIME_COL       = 'years_from_BL_exact'

# --- Cox covariates ---------------------------------------------------------
CORE_CONTINUOUS_COVARIATES  = ['age_at_visit']
CORE_CATEGORICAL_COVARIATES = ['SEX']
CORE_ALPHA = 0.05

# --- Kinetics ---------------------------------------------------------------
# 'P237' / 'P299' / 'P155': that project's own columns from the wide baseline
#     file, so every participant with data from that assay is kept.
# 'selected': SAA_*_BL (whatever the pipeline chose), keeping only participants
#     whose chosen project == CORE_KINETICS_PROJECT.
# Kinetics are never pooled across projects: a 24-hour, a 35-hour and a
# 150-hour run do not share a scale.
CORE_KINETICS_SOURCE  = 'P237'
CORE_KINETICS_PROJECT = 237       # used when CORE_KINETICS_SOURCE == 'selected'
CORE_AGG              = 'mean'    # replicate aggregation of the P* columns
CORE_METRICS = ['Fmax', 'TTT', 'AUC', 'TimeToMaxSlope', 'MaxSlope']

# --- Figure: x axis (PLOT ONLY -- changes what you see, never what is tested)
K5_XMAX  = 12          # x-axis limit in years. None = observed maximum.
K5_XSTEP = None        # tick spacing in years. None = chosen from K5_XMAX.

# --- Censoring (DATA -- changes the analysis) -------------------------------
K5_ADMIN_CENSOR   = None    # e.g. 10 to censor the DATA at 10 years
K5_CENSOR_AT_XMAX = True    # True -> censor the data at K5_XMAX, so log-rank and
                            # Cox describe exactly the curve that is drawn.
                            # Anyone who converts after K5_XMAX is counted as
                            # event-free at K5_XMAX, so the event count drops.

K5_MIN_NONMISSING = 40      # a parameter needs this many values to be drawn

# (metric, EXPOSURE label, REFERENCE label), median split
K5_PARAMS = [
    ('Fmax', 'High', 'Low'),
    ('TTT',  'Fast', 'Slow'),
    ('AUC',  'High', 'Low'),
]
K5_ABOVE_MEDIAN_LABELS = {'High', 'Slow'}   # labels that denote > median

# Grid position -> content: 'saa_overall', a parameter name, 'forest', or None
# for a blank cell.
K5_PANEL_ORDER = ['saa_overall', 'forest', None, 'Fmax', 'TTT', 'AUC']

K5_CURVE = 'survival'      # 'survival' (falling) or 'incidence' (rising, 1-KM)
K5_OUTCOME_LABEL = 'auto'  # or any string, e.g. 'Conversion-free survival'
K5_CREF, K5_CEXP = '#378ADD', '#D85A30'           # reference / exposure (kinetics)
K5_CSAA_NEG, K5_CSAA_POS = '#1D9E75', '#C0392B'   # panel A: SAA-negative / SAA-positive
K5_SHOW_CI     = True
K5_AT_RISK     = True
K5_SHOW_HR_BOX = False     # True -> adjusted HR box inside each KM panel
K5_MIN_ARM_N   = 2         # an arm smaller than this is not drawn or tabulated
K5_HR_FLOOR    = 0.02      # clamp for the forest x-axis against separated CIs
K5_DPI         = 600
# ---------------------------- END OF CONFIG ---------------------------------

## Sample, models and figure

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings('ignore')

CORE_BASE = Path(BASE_DIR)
CORE_SRC  = CORE_BASE / 'results_baseline' / 'prodromal_saa_longitudinal.csv'
CORE_WIDE = CORE_BASE / 'results_baseline' / 'prodromal_saa_baseline_core.csv'
CORE_OUT  = CORE_BASE / 'analysis'
if not CORE_SRC.exists():
    raise FileNotFoundError(f'Missing input: {CORE_SRC}  (run 01_build_dataset first)')

CORE_ASSAY_NAME = {'P155': 'Project 155 (150-hour pilot assay)',
                   'P237': 'Project 237 (24-hour assay)',
                   'P299': 'Project 299 (35-hour assay)'}


def core_source_name():
    if CORE_KINETICS_SOURCE == 'selected':
        return f'Project {CORE_KINETICS_PROJECT} (pipeline-selected kinetics)'
    return CORE_ASSAY_NAME.get(CORE_KINETICS_SOURCE, CORE_KINETICS_SOURCE)


# ============================================================================
# Sample: baseline set and survival set
# Event = first visit with PRIMDIAG in CORE_EVENT_PRIMDIAG; otherwise censored
# at the last observed visit.
# ============================================================================
def core_build():
    d = pd.read_csv(CORE_SRC, na_values=[''], low_memory=False)
    n0_rows, n0_pat = len(d), d.PATNO.nunique()
    if CORE_ENROLL_YEAR_MAX is not None:
        d = d[d.BL_enroll_year <= CORE_ENROLL_YEAR_MAX]
    d = d.sort_values(['PATNO', 'visit_n'])

    bl = d[d.is_baseline == 1].set_index('PATNO').copy()
    bl['n_visits_obs'] = d.groupby('PATNO').size().reindex(bl.index)

    first_ev = d[d.PRIMDIAG.isin(CORE_EVENT_PRIMDIAG)].groupby('PATNO')[CORE_TIME_COL].min()
    last_t = d.groupby('PATNO')[CORE_TIME_COL].max()
    bl['event'] = bl.index.isin(first_ev.index).astype(int)
    bl['time'] = np.where(bl.event == 1, first_ev.reindex(bl.index).values,
                          last_t.reindex(bl.index).values)
    bl['prevalent_PD'] = bl.PRIMDIAG.isin(CORE_EVENT_PRIMDIAG).astype(int)

    surv = bl.copy()
    steps = [('baseline rows after enrollment filter', len(surv))]
    if CORE_EXCLUDE_PREVALENT:
        surv = surv[surv.prevalent_PD == 0]
        steps.append(('minus prevalent cases at baseline', len(surv)))
    if CORE_REQUIRE_FOLLOWUP:
        surv = surv[surv.n_visits_obs > 1]
        steps.append(('minus baseline-only participants', len(surv)))
    surv = surv[surv.time.notna() & (surv.time > 0)]
    steps.append(('minus non-positive follow-up time', len(surv)))
    return bl, surv, steps, (n0_rows, n0_pat)


def core_kinetics(df):
    """Restrict df to participants with kinetics from the chosen assay and make
    SAA_<metric>_BL hold that assay's values."""
    out = df.copy()
    cols = [f'SAA_{m}_BL' for m in CORE_METRICS]
    if CORE_KINETICS_SOURCE == 'selected':
        proj = pd.to_numeric(out.SAA_Kinetics_Project_BL, errors='coerce')
        return out[proj == CORE_KINETICS_PROJECT]
    if not CORE_WIDE.exists():
        raise FileNotFoundError(
            f'{CORE_WIDE} not found. It holds the per-project P155_/P237_/P299_ '
            "columns. Either rerun 01_build_dataset, or set CORE_KINETICS_SOURCE = 'selected'.")
    w = pd.read_csv(CORE_WIDE, na_values=[''], low_memory=False).set_index('PATNO')
    missing = [f'{CORE_KINETICS_SOURCE}_{m}_{CORE_AGG}' for m in CORE_METRICS
               if f'{CORE_KINETICS_SOURCE}_{m}_{CORE_AGG}' not in w.columns]
    if missing:
        raise KeyError(f'{CORE_WIDE.name} has no columns {missing}. '
                       'Check CORE_KINETICS_SOURCE / CORE_AGG.')
    for m, c in zip(CORE_METRICS, cols):
        out[c] = pd.to_numeric(
            w[f'{CORE_KINETICS_SOURCE}_{m}_{CORE_AGG}'].reindex(out.index), errors='coerce')
    return out[out[cols].notna().any(axis=1)]


CORE_BL, CORE_SURV, CORE_STEPS, CORE_SOURCE_SIZE = core_build()
CORE_KIN_SURV = core_kinetics(CORE_SURV[CORE_SURV.SAA_Status_BL == 'Positive'])

print('Sample definition')
print(f'  source file: {CORE_SOURCE_SIZE[0]} rows, {CORE_SOURCE_SIZE[1]} participants')
for name, n in CORE_STEPS:
    print(f'  {name:<44} {n}')
print(f'  Survival set: n = {len(CORE_SURV)}  ({int(CORE_SURV.event.sum())} events, '
      f'median follow-up {CORE_SURV.time.median():.2f} y)')
print(f'  SAA-positive with {core_source_name()} kinetics ({CORE_AGG} of replicates): '
      f'n = {len(CORE_KIN_SURV)} ({int(CORE_KIN_SURV.event.sum())} events)\n')


# ============================================================================
# Survival analysis and figure
#   A  Overall cohort: KM by SAA status (SAA-positive vs SAA-negative)
#   B  Forest plot of adjusted HRs (SAA status; Fmax, TTT, AUC)
#   D-F KM by median split of Fmax, TTT, AUC (SAA-positive, chosen assay)
# ============================================================================
K5_FIG = CORE_OUT / 'figures'
K5_FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 600, 'savefig.dpi': K5_DPI, 'savefig.bbox': 'tight',
                     'font.size': 12, 'axes.titlesize': 12, 'axes.labelsize': 12,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.22, 'grid.linewidth': 0.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})


def k5_source_name():
    return core_source_name()


def k5_column(metric):
    """CORE already wrote the chosen assay's values into SAA_<metric>_BL."""
    return f'SAA_{metric}_BL'


def k5_p(p):
    if p is None or not np.isfinite(p):
        return 'n/a'
    return '<0.001' if p < 0.001 else f'{p:.3f}'


def k5_pstr(p):
    s = k5_p(p)
    return 'P' + s if s.startswith('<') else 'P = ' + s


def k5_stars(p):
    if p is None or not np.isfinite(p):
        return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''


def k5_hr(hr, lo, hi):
    return f'{hr:.2f} ({lo:.2f}–{hi:.2f})' if np.isfinite(hr) else 'not estimable'


def k5_outcome_label():
    if K5_OUTCOME_LABEL != 'auto':
        return K5_OUTCOME_LABEL
    codes = sorted(CORE_EVENT_PRIMDIAG)
    if not all(c in CORE_DIAG_NAMES for c in codes):
        stem = 'Phenoconversion'
    elif codes == [1, 5, 11]:
        stem = 'Synucleinopathy'
    else:
        stem = '/'.join(CORE_DIAG_NAMES[c] for c in codes)
    return (f'Phenoconversion proportion ({stem})' if K5_CURVE == 'incidence'
            else 'Phenoconversion-free survival')


def k5_design(df, extra):
    """Cox design matrix.

    A participant with a MISSING CATEGORICAL covariate must be dropped, not
    absorbed into the reference level. pd.get_dummies() drops NaN by default,
    which leaves such a participant as all-zeros -- indistinguishable from the
    reference group and invisible to .dropna(). The `bad` mask below removes
    them explicitly, so categorical and continuous missingness behave alike.
    """
    cols = dict(extra)
    bad = pd.Series(False, index=df.index)

    for c in CORE_CONTINUOUS_COVARIATES:
        if c in df:
            cols[c] = pd.to_numeric(df[c], errors='coerce')

    for c in CORE_CATEGORICAL_COVARIATES:
        if c not in df:
            continue
        s = df[c].astype('object')
        miss = s.isna() | s.astype(str).str.strip().isin(['', '.', 'nan', 'None', 'NA'])
        bad |= miss
        dm = pd.get_dummies(s.where(~miss), prefix=c, drop_first=True, dtype=float)
        for k in dm:
            cols[k] = dm[k]

    X = pd.DataFrame(cols, index=df.index)
    X['time'] = pd.to_numeric(df['time'], errors='coerce')
    X['event'] = pd.to_numeric(df['event'], errors='coerce')
    return X[~bad].dropna()


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
k5_s = CORE_KIN_SURV.copy()

k5_censor = float(K5_XMAX) if (K5_CENSOR_AT_XMAX and K5_XMAX is not None) else K5_ADMIN_CENSOR
if k5_censor is not None:
    k5_s.loc[k5_s.time > k5_censor, 'event'] = 0
    k5_s.loc[k5_s.time > k5_censor, 'time'] = float(k5_censor)

k5_xmax = float(np.ceil(k5_s.time.max())) if K5_XMAX is None else float(K5_XMAX)
k5_step = K5_XSTEP if K5_XSTEP is not None else (1.0 if k5_xmax <= 6 else
                                                 2.0 if k5_xmax <= 14 else 5.0)
k5_grid = np.arange(0, k5_xmax + 1e-9, k5_step)

print(f'Kinetics source: {k5_source_name()}'
      + ('' if CORE_KINETICS_SOURCE == 'selected' else f', {CORE_AGG} of replicates'))
print(f'SAA-positive with these kinetics: {len(k5_s)} participants, '
      f'{int(k5_s.event.sum())} events, median follow-up {k5_s.time.median():.2f} y, '
      f'max {k5_s.time.max():.1f} y')
print(f'x-axis: 0-{k5_xmax:g} y, ticks every {k5_step:g} y'
      + ('' if k5_censor is None else f'; DATA censored at {k5_censor:g} y'))
if k5_censor is None and k5_s.time.max() > k5_xmax:
    print(f'  NOTE: curves are cut at {k5_xmax:g} y for display, but log-rank and Cox '
          f'use the full {k5_s.time.max():.1f} y of follow-up.\n'
          f'        Set K5_CENSOR_AT_XMAX = True to make the tests match the picture.')
if abs(CORE_ALPHA - 0.05) > 1e-9:
    print(f'  NOTE: significance colouring uses alpha = {CORE_ALPHA}, but the * ** *** '
          f'annotations keep the conventional 0.05 / 0.01 / 0.001 thresholds.')

# ---------------------------------------------------------------------------
# Estimates -- median split, log-rank, adjusted Cox
# ---------------------------------------------------------------------------
k5_est, k5_recs = {}, []
for name, expo, ref in K5_PARAMS:
    col = k5_column(name)
    if col not in k5_s.columns or k5_s[col].notna().sum() < K5_MIN_NONMISSING:
        print(f'  skipped {name}: {0 if col not in k5_s.columns else int(k5_s[col].notna().sum())} '
              f'values (< {K5_MIN_NONMISSING})')
        continue
    above = expo if expo in K5_ABOVE_MEDIAN_LABELS else ref
    below = ref if expo in K5_ABOVE_MEDIAN_LABELS else expo

    med = float(k5_s[col].median())
    d = k5_s[k5_s[col].notna()].copy()
    d['ksplit'] = np.where(d[col] > med, above, below)   # ties go to the low arm
    d['exposure'] = (d.ksplit == expo).astype(float)
    arm_e, arm_r = d[d.ksplit == expo], d[d.ksplit == ref]
    if min(len(arm_e), len(arm_r)) < K5_MIN_ARM_N:
        print(f'  skipped {name}: median split gives arms of {len(arm_e)} / {len(arm_r)}')
        continue

    try:
        lr = logrank_test(arm_e.time, arm_r.time, arm_e.event, arm_r.event).p_value
    except Exception:
        lr = np.nan

    # The Cox sample is NOT the KM sample: rows missing any covariate are gone.
    X = k5_design(d, {'exposure': d.exposure})
    hr = lo = hi = p = np.nan
    n_cox, ev_cox, n_par = int(X.shape[0]), int(X.event.sum()), X.shape[1] - 2
    try:
        srow = CoxPHFitter().fit(X, duration_col='time', event_col='event').summary.loc['exposure']
        hr, lo, hi, p = (float(srow['exp(coef)']), float(srow['exp(coef) lower 95%']),
                         float(srow['exp(coef) upper 95%']), float(srow['p']))
    except Exception as e:
        print(f'  Cox failed for {name}: {type(e).__name__}')

    k5_est[name] = {'name': name, 'col': col, 'exposure': expo, 'reference': ref,
                    'median_cut': med, 'data': d, 'arm_exp': arm_e, 'arm_ref': arm_r,
                    'logrank_p': lr, 'hr': hr, 'lo': lo, 'hi': hi, 'p': p,
                    'n': len(d), 'events': int(d.event.sum()),
                    'n_cox': n_cox, 'events_cox': ev_cox}
    k5_recs.append({'parameter': name, 'contrast': f'{expo} vs {ref}', 'reference': ref,
                    'median_cut': med, 'n': len(d), 'events': int(d.event.sum()),
                    'n_exposure': len(arm_e), 'n_reference': len(arm_r),
                    'events_exposure': int(arm_e.event.sum()),
                    'events_reference': int(arm_r.event.sum()),
                    'n_cox': n_cox, 'events_cox': ev_cox,
                    'n_dropped_missing_covariate': len(d) - n_cox,
                    'HR': hr, 'CI_low': lo, 'CI_high': hi, 'p': p,
                    'logrank_p': lr, 'events_per_parameter': round(ev_cox / max(n_par, 1), 1)})

k5_forest = [k5_est[n] for n, _, _ in K5_PARAMS if n in k5_est]
if not k5_est:
    raise RuntimeError('No kinetic parameter had enough data to draw -- '
                       'check CORE_KIN_SURV and K5_MIN_NONMISSING.')

# ---------------------------------------------------------------------------
# Panel A -- overall cohort, SAA-positive vs SAA-negative (same sample as Table 2)
# Censored at the same horizon as the kinetics panels so the test matches the
# drawn curve.
# ---------------------------------------------------------------------------
k5_saa = CORE_SURV[CORE_SURV.SAA_Status_BL.isin(['Positive', 'Negative'])].copy()
if k5_censor is not None:
    k5_saa.loc[k5_saa.time > k5_censor, 'event'] = 0
    k5_saa.loc[k5_saa.time > k5_censor, 'time'] = float(k5_censor)
k5_saa_pos = k5_saa[k5_saa.SAA_Status_BL == 'Positive']
k5_saa_neg = k5_saa[k5_saa.SAA_Status_BL == 'Negative']
try:
    k5_saa_lr = logrank_test(k5_saa_pos.time, k5_saa_neg.time,
                             k5_saa_pos.event, k5_saa_neg.event).p_value
except Exception:
    k5_saa_lr = np.nan
print(f'Panel A (overall): {len(k5_saa)} participants, {int(k5_saa.event.sum())} events, '
      f'SAA+ {len(k5_saa_pos)} / SAA- {len(k5_saa_neg)}, log-rank {k5_pstr(k5_saa_lr)}')

# --- ADDED: Adjusted Cox for panel A (SAA+ vs SAA-), same covariates as kinetics,
# added as the TOP row of the forest plot and the first row of the CSV.
k5_saa['exposure'] = (k5_saa.SAA_Status_BL == 'Positive').astype(float)
X_saa = k5_design(k5_saa, {'exposure': k5_saa.exposure})
s_hr = s_lo = s_hi = s_p = np.nan
try:
    srow = CoxPHFitter().fit(X_saa, duration_col='time',
                             event_col='event').summary.loc['exposure']
    s_hr, s_lo, s_hi, s_p = (float(srow['exp(coef)']), float(srow['exp(coef) lower 95%']),
                             float(srow['exp(coef) upper 95%']), float(srow['p']))
except Exception as e:
    print(f'  Cox failed for SAA status: {type(e).__name__}')

k5_saa_est = {'name': 'SAA status', 'exposure': 'SAA-positive', 'reference': 'SAA-negative',
              'hr': s_hr, 'lo': s_lo, 'hi': s_hi, 'p': s_p,
              'n_cox': int(X_saa.shape[0]), 'events_cox': int(X_saa.event.sum())}
k5_forest = [k5_saa_est] + k5_forest          # SAA as the top row

k5_recs.insert(0, {'parameter': 'SAA_status', 'contrast': 'Positive vs Negative',
                   'reference': 'Negative', 'median_cut': np.nan,
                   'n': len(k5_saa), 'events': int(k5_saa.event.sum()),
                   'n_exposure': len(k5_saa_pos), 'n_reference': len(k5_saa_neg),
                   'events_exposure': int(k5_saa_pos.event.sum()),
                   'events_reference': int(k5_saa_neg.event.sum()),
                   'n_cox': int(X_saa.shape[0]), 'events_cox': int(X_saa.event.sum()),
                   'n_dropped_missing_covariate': len(k5_saa) - int(X_saa.shape[0]),
                   'HR': s_hr, 'CI_low': s_lo, 'CI_high': s_hi, 'p': s_p,
                   'logrank_p': k5_saa_lr,
                   'events_per_parameter': round(int(X_saa.event.sum()) /
                                                 max(X_saa.shape[1] - 2, 1), 1)})
print(f'Panel A adjusted Cox: HR {k5_hr(s_hr, s_lo, s_hi)} {k5_pstr(s_p)} '
      f'(n={int(X_saa.shape[0])}, events={int(X_saa.event.sum())})')


# ---------------------------------------------------------------------------
# Drawing
# ---------------------------------------------------------------------------
def k5_low_at_cap(kmf):
    """Lowest survival (curve or CI band) still VISIBLE at the x-axis limit.

    Using the last point of the whole curve instead would stretch the y axis
    down to a value that is off-screen whenever K5_XMAX truncates the follow-up,
    squashing every visible curve into the top of the panel."""
    cap = min(k5_xmax, float(kmf.timeline.max()))
    low = float(kmf.predict(cap))
    if K5_SHOW_CI:
        ci = kmf.confidence_interval_
        idx = ci.index[ci.index <= cap]
        if len(idx):
            low = min(low, float(ci.loc[idx[-1], ci.columns[0]]))
    return low


def k5_draw_km(ax_km, ax_risk, e, cref=None, cexp=None):
    ylo_seen = []
    cref = K5_CREF if cref is None else cref
    cexp = K5_CEXP if cexp is None else cexp
    # reference first in the legend; arms too small to plot are dropped here so
    # the legend and the numbers-at-risk table can never disagree
    order = [(arm, lbl, colr) for arm, lbl, colr in
             [(e['arm_ref'], e['reference'], cref), (e['arm_exp'], e['exposure'], cexp)]
             if len(arm) >= K5_MIN_ARM_N]
    for arm, lbl, colr in order:
        kmf = KaplanMeierFitter()
        kmf.fit(arm.time, arm.event, label=f'{lbl} (n={len(arm)}, events={int(arm.event.sum())})')
        if K5_CURVE == 'incidence':
            kmf.plot_cumulative_density(ax=ax_km, ci_show=K5_SHOW_CI, color=colr, lw=1.5,
                                        ci_alpha=0.13, censor_styles={'ms': 3})
        else:
            kmf.plot_survival_function(ax=ax_km, ci_show=K5_SHOW_CI, color=colr, lw=1.5,
                                       ci_alpha=0.13, censor_styles={'ms': 3})
        ylo_seen.append(k5_low_at_cap(kmf))   # measured at the x limit, not at the end

    ax_km.set_xlim(0, k5_xmax)
    ax_km.set_xticks(k5_grid)
    ax_km.set_xlabel('')          # lifelines writes 'timeline' by default
    ax_km.set_title(f"{e['name']}: {e['exposure']} vs {e['reference']} (ref)\n"
                    f"log-rank {k5_pstr(e['logrank_p'])}", fontsize=12)
    ax_km.set_ylabel(k5_outcome_label(), fontsize=12)
    ax_km.legend(fontsize=12, loc='lower left' if K5_CURVE == 'survival' else 'upper left',
                 frameon=False)
    if K5_SHOW_HR_BOX:
        ax_km.text(0.985, 0.03 if K5_CURVE == 'incidence' else 0.965,
                   f"Adjusted HR {k5_hr(e['hr'], e['lo'], e['hi'])}\n"
                   f"{k5_pstr(e['p'])}{k5_stars(e['p'])}",
                   transform=ax_km.transAxes, ha='right',
                   va='bottom' if K5_CURVE == 'incidence' else 'top', fontsize=12,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                             edgecolor='#CCCBC6', linewidth=0.7))

    # numbers at risk, in their own axes underneath
    ax_risk.set_xlim(0, k5_xmax)
    ax_risk.set_ylim(0, 1)
    ax_risk.set_yticks([]); ax_risk.set_xticks([])
    ax_risk.grid(False)
    for sp in ax_risk.spines.values():
        sp.set_visible(False)
    ax_risk.set_xlabel('Years from baseline')
    if K5_AT_RISK:
        ax_risk.text(0, 0.93, 'Number at risk', fontsize=10, color='#5F5E5A',
                     ha='left', va='top')
        for j, (arm, lbl, colr) in enumerate(order):
            y = 0.52 - 0.34 * j
            ax_risk.text(-0.012, y, lbl, transform=ax_risk.transAxes, fontsize=10,
                         color=colr, ha='right', va='center')
            for t in k5_grid:
                ha = 'left' if t == k5_grid[0] else 'right' if t == k5_grid[-1] else 'center'
                ax_risk.text(t, y, str(int((arm.time >= t).sum())), fontsize=10,
                             color=colr, ha=ha, va='center')
    return ylo_seen


def k5_draw_saa(ax_km, ax_risk):
    """Panel A: overall cohort KM by SAA status, in the same style as the
    kinetics panels (reference = SAA-negative), green vs red. It sets its own y range
    and is NOT added to k5_km_axes, so the kinetics panels keep their y axis."""
    e = {'name': 'Overall cohort', 'exposure': 'SAA-positive', 'reference': 'SAA-negative',
         'arm_exp': k5_saa_pos, 'arm_ref': k5_saa_neg, 'logrank_p': k5_saa_lr,
         'hr': s_hr, 'lo': s_lo, 'hi': s_hi, 'p': s_p}
    ylo = k5_draw_km(ax_km, ax_risk, e, cref=K5_CSAA_NEG, cexp=K5_CSAA_POS)
    if ylo:
        if K5_CURVE == 'incidence':
            ax_km.set_ylim(0, min(1.02, float(np.ceil((1 - min(ylo) + 0.06) * 20) / 20)))
        else:
            ax_km.set_ylim(max(0.0, float(np.floor((min(ylo) - 0.06) * 20) / 20)), 1.02)


def k5_draw_forest(ax):
    ax.grid(axis='y', alpha=0)
    ys = np.arange(len(k5_forest))[::-1]
    fin = [r for r in k5_forest if np.isfinite(r['hr'])]
    lo_all = max(min([r['lo'] for r in fin] + [0.5]) if fin else 0.5, K5_HR_FLOOR)
    hi_all = max([r['hi'] for r in fin] + [2.0]) if fin else 2.0
    span = np.log10(hi_all / lo_all)
    x_text, x_right = hi_all * 10 ** (span * 0.08), hi_all * 10 ** (span * 1.25)
    for y, r in zip(ys, k5_forest):
        if not np.isfinite(r['hr']):
            ax.text(x_text, y, 'not estimable', va='center', fontsize=9, color='#A32D2D')
            continue
        col = '#185FA5' if (np.isfinite(r['p']) and r['p'] < CORE_ALPHA) else '#5F5E5A'
        ax.plot([max(r['lo'], lo_all), r['hi']], [y, y], color=col, lw=1.5, solid_capstyle='butt')
        ax.plot(r['hr'], y, 'o', color=col, ms=5.5)
        ax.text(x_text, y, f"{k5_hr(r['hr'], r['lo'], r['hi'])}\n"
                           f"{k5_pstr(r['p'])}{k5_stars(r['p'])}",
                va='center', fontsize=12, color=col)
    ax.axvline(1.0, color='#888780', ls='--', lw=0.8, zorder=0)
    ax.set_xscale('log')
    ax.set_xlim(lo_all / 1.4, x_right)
    ax.set_ylim(-0.7, len(k5_forest) - 0.3)
    ax.set_yticks(ys)
    # n per row = the Cox sample for THAT parameter, which is what the HR used
    ax.set_yticklabels([f"{r['name']}" for r in k5_forest], fontsize=12)
    cand = [0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32]
    ticks = [t for t in cand if lo_all / 1.4 <= t <= hi_all * 1.05]
    while len(ticks) > 6:
        ticks = [t for j, t in enumerate(ticks) if t == 1 or j % 2 == 0]
    if ticks:
        ax.set_xticks(ticks); ax.set_xticklabels(['%g' % t for t in ticks])
    ax.minorticks_off()
    ax.set_xlabel('Adjusted HR for phenoconversion (95% CI)')

    # ADDED: SAA row (whole cohort) is separated from the kinetics rows (SAA+ only)
    has_saa = bool(k5_forest) and k5_forest[0]['name'] == 'SAA status'
    kin = k5_forest[1:] if has_saa else k5_forest
    if has_saa and len(k5_forest) > 1:
        ax.axhline(ys[0] - 0.5, color='#CCCBC6', lw=0.7)   # SAA row | kinetics rows

    # one honest header line: the Cox models, not the KM curves, are summarised
    ns = [r['n_cox'] for r in kin]
    evs = [r['events_cox'] for r in kin]
    n_lab = f'{min(ns)}' if min(ns) == max(ns) else f'{min(ns)}–{max(ns)}'
    e_lab = f'{min(evs)}' if min(evs) == max(evs) else f'{min(evs)}–{max(evs)}'
    if has_saa:
        ax.set_title(f'Adjusted HRs\n(SAA: n={k5_forest[0]["n_cox"]}; '
                     f'kinetics, SAA+ only: n={n_lab})', fontsize=12)
    else:
        ax.set_title(f'Median-split kinetic parameters \n (Cox n={n_lab}, events={e_lab})', fontsize=12)


k5_fig = plt.figure(figsize=(17.4, 11.8))
k5_gs = k5_fig.add_gridspec(2, 3, hspace=0.40, wspace=0.30)
k5_letters = ['A', 'B', 'C', 'D', 'E', 'F']
k5_km_axes, k5_ylo = [], []

for i, key in enumerate(K5_PANEL_ORDER[:6]):
    cell = k5_gs[i // 3, i % 3]
    if key == 'forest':
        ax = k5_fig.add_subplot(cell)
        k5_draw_forest(ax)
    elif key == 'saa_overall':
        inner = cell.subgridspec(2, 1, height_ratios=[4.4, 1.0], hspace=0.30)
        ax = k5_fig.add_subplot(inner[0])
        ax_risk = k5_fig.add_subplot(inner[1])
        k5_draw_saa(ax, ax_risk)          # NOT added to k5_km_axes / k5_ylo
    elif key in k5_est:
        inner = cell.subgridspec(2, 1, height_ratios=[4.4, 1.0], hspace=0.30)
        ax = k5_fig.add_subplot(inner[0])
        ax_risk = k5_fig.add_subplot(inner[1])
        k5_ylo += k5_draw_km(ax, ax_risk, k5_est[key])
        k5_km_axes.append(ax)
    else:
        continue
    ax.text(-0.16, 1.06, k5_letters[i], transform=ax.transAxes, fontsize=14,
            fontweight='bold', va='bottom', ha='left')

# one y range for all kinetics KM panels so the parameters stay comparable
if k5_ylo:
    if K5_CURVE == 'incidence':
        k5_top = min(1.02, float(np.ceil((1 - min(k5_ylo) + 0.06) * 20) / 20))
        for ax in k5_km_axes:
            ax.set_ylim(0, k5_top)
    else:
        k5_bottom = max(0.0, float(np.floor((min(k5_ylo) - 0.06) * 20) / 20))
        for ax in k5_km_axes:
            ax.set_ylim(k5_bottom, 1.02)

k5_fig.suptitle(f'{k5_outcome_label()} by median-split kinetic parameter — '
                f'SAA-positive, {k5_source_name()}', fontsize=14, y=0.985)

k5_stem = K5_FIG / 'fig4b_KM_kinetics_median_split_with_forest'
k5_fig.savefig(k5_stem.with_suffix('.png'), dpi=K5_DPI)
k5_fig.savefig(k5_stem.with_suffix('.pdf'))
try:
    k5_fig.savefig(k5_stem.with_suffix('.tiff'), dpi=K5_DPI,
                   pil_kwargs={'compression': 'tiff_lzw'})
except Exception as e:
    print('TIFF export skipped:', e)
plt.close(k5_fig)

K5_EST_COLS = ['parameter', 'contrast', 'reference', 'median_cut', 'n', 'events',
               'n_exposure', 'n_reference', 'events_exposure', 'events_reference',
               'n_cox', 'events_cox', 'n_dropped_missing_covariate',
               'HR', 'CI_low', 'CI_high', 'p', 'logrank_p', 'events_per_parameter']
pd.DataFrame(k5_recs, columns=K5_EST_COLS).to_csv(
    K5_FIG / 'fig4b_kinetics_median_split_estimates.csv', index=False)

print()
for r in k5_recs:
    drop = r['n_dropped_missing_covariate']
    print(f"  {r['parameter']:<16} cut={r['median_cut']:<12.4g} {r['contrast']:<14} "
          f"arms {r['n_exposure']}/{r['n_reference']:<6} "
          f"HR {k5_hr(r['HR'], r['CI_low'], r['CI_high']):<22} {k5_pstr(r['p'])}  "
          f"log-rank {k5_pstr(r['logrank_p'])}"
          + (f"   [Cox n={r['n_cox']}, {drop} dropped for missing covariates]" if drop else ''))
print(f"\nTests use follow-up to "
      + (f"{k5_censor:g} y (administratively censored)" if k5_censor is not None
         else f"{k5_s.time.max():.1f} y; the panels are drawn to {k5_xmax:g} y")
      + f". P-values are unadjusted for the {len(k5_recs)} parameters tested.")
print('Panel key: ' + ', '.join(f'{k5_letters[i]} = {k}'
                                for i, k in enumerate(K5_PANEL_ORDER[:6])
                                if k in ('forest', 'saa_overall') or k in k5_est))
print('written to', k5_stem.with_suffix('.png'))